# CycleGAN — unpaired translation via cycle-consistency

> Tutorial pair for [`cyclegan.py`](cyclegan.py).

## 1. Intuition
We want to translate between two domains (horses<->zebras, photos<->paintings)
but have **no aligned pairs**. Adversarial loss alone is too weak: a generator
could send every input to one realistic target sample and still fool the critic.
The fix is **cycle-consistency** — translate $X\to Y\to X$ and you must get the
original back. That closed loop forces the mapping to preserve content, turning
an underconstrained problem into a well-posed one.

## 2. Concept (the slide)
- **Two generators:** $G:X\to Y$ and $F:Y\to X$.
- **Two discriminators:** $D_Y$ scores $Y$-realism, $D_X$ scores $X$-realism.
- **Cycle-consistency:** $F(G(x))\approx x$ and $G(F(y))\approx y$ (L1).
- **Identity loss:** $G(y)\approx y$, $F(x)\approx x$ to preserve color/scale.
- We use the **least-squares** adversarial loss (LSGAN), which CycleGAN found
  more stable than BCE. Here both domains are tiny 2-D point clouds related by a
  fixed affine transform, so we can *measure* whether translation succeeded.

## 3. Math derivation — adversarial + cycle + identity

**Adversarial (LSGAN form), e.g. for $G$ and $D_Y$:**
$$\mathcal L_{\text{GAN}}(G,D_Y)=\mathbb E_{y}\big[(D_Y(y)-1)^2\big]
 +\mathbb E_{x}\big[(D_Y(G(x)))^2\big],$$
with $G$ trying to push $D_Y(G(x))\to 1$. The same pair of terms applies to
$F$ and $D_X$. This alone only asks "does the output *look* like the target
domain?" — it says nothing about **which** input produced it.

**Cycle-consistency.** Add the constraint that round-trips are identity:
$$\mathcal L_{\text{cyc}}(G,F)=\mathbb E_{x}\big[\lVert F(G(x))-x\rVert_1\big]
 +\mathbb E_{y}\big[\lVert G(F(y))-y\rVert_1\big].$$
L1 (not L2) is used because it encourages sharp, less-blurry reconstructions.
This term is what prevents **mode collapse** to a single target: if many inputs
mapped to the same $y$, $F$ could not invert it to recover each distinct $x$.

**Identity.** Optionally regularize with
$$\mathcal L_{\text{id}}=\mathbb E_{y}\big[\lVert G(y)-y\rVert_1\big]
 +\mathbb E_{x}\big[\lVert F(x)-x\rVert_1\big],$$
which anchors color/scale (a sample already in the target domain should pass
through unchanged).

**Full objective.**
$$\min_{G,F}\max_{D_X,D_Y}\;\mathcal L_{\text{GAN}}(G,D_Y)+\mathcal L_{\text{GAN}}(F,D_X)
 +\lambda_{\text{cyc}}\,\mathcal L_{\text{cyc}}+\lambda_{\text{id}}\,\mathcal L_{\text{id}}.$$

**Why it differs from vanilla GAN.** A vanilla GAN matches one distribution from
noise. CycleGAN couples **two** GANs through an L1 cycle term, replacing the need
for paired supervision with a structural invertibility constraint.

## 4. Generator / key component

In [ ]:
# ===== actual implementation from cyclegan.py =====
from __future__ import annotations

import numpy as np

import torch

import torch.nn as nn

SEED = 0

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

_THETA = np.pi / 2

_ROT = np.array([[np.cos(_THETA), -np.sin(_THETA)],
                 [np.sin(_THETA), np.cos(_THETA)]], dtype=np.float32)

_SCALE = np.float32(1.5)

_SHIFT = np.array([2.0, -1.0], dtype=np.float32)

def make_domain_x(n: int = 1024, seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return (rng.normal(size=(n, 2)) * np.array([1.0, 0.4], np.float32)).astype(np.float32)

def make_domain_y(n: int = 1024, seed: int = SEED + 7) -> np.ndarray:
    """Domain Y = affine(X) on *independently* drawn samples (so it is unpaired)."""
    x = make_domain_x(n, seed)
    return (_SCALE * (x @ _ROT.T) + _SHIFT).astype(np.float32)

class Generator(nn.Module):
    """Maps a point from one domain to the other (residual-style)."""

    def __init__(self, dim: int = 2, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

## 5. Trainer / losses

In [ ]:
# ===== actual implementation from cyclegan.py =====
class Discriminator(nn.Module):
    """Scores whether a point belongs to its target domain (LSGAN: unbounded)."""

    def __init__(self, dim: int = 2, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2, True),
            nn.Linear(hidden, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class CycleGANTorch:
    def __init__(self, dim: int = 2, lr: float = 2e-4,
                 lambda_cyc: float = 10.0, lambda_id: float = 1.0):
        torch.manual_seed(SEED)
        self.dev = get_device()
        self.lambda_cyc, self.lambda_id = lambda_cyc, lambda_id
        self.G = Generator(dim).to(self.dev)   # X -> Y
        self.F = Generator(dim).to(self.dev)   # Y -> X
        self.D_Y = Discriminator(dim).to(self.dev)
        self.D_X = Discriminator(dim).to(self.dev)
        self.optG = torch.optim.Adam(
            list(self.G.parameters()) + list(self.F.parameters()),
            lr=lr, betas=(0.5, 0.999))
        self.optD = torch.optim.Adam(
            list(self.D_X.parameters()) + list(self.D_Y.parameters()),
            lr=lr, betas=(0.5, 0.999))
        self.l1 = nn.L1Loss()

    @staticmethod
    def _mse(out: torch.Tensor, target: float) -> torch.Tensor:
        return ((out - target) ** 2).mean()

    def fit(self, X: np.ndarray, Y: np.ndarray, steps: int = 800, batch: int = 128):
        X = torch.as_tensor(X, dtype=torch.float32, device=self.dev)
        Y = torch.as_tensor(Y, dtype=torch.float32, device=self.dev)
        self.g_hist, self.cyc_hist = [], []
        for _ in range(steps):
            x = X[torch.randint(0, len(X), (batch,), device=self.dev)]
            y = Y[torch.randint(0, len(Y), (batch,), device=self.dev)]

            # --- Generators: adversarial + cycle + identity ---
            fake_y = self.G(x)              # X -> Y
            fake_x = self.F(y)              # Y -> X
            rec_x = self.F(fake_y)          # X -> Y -> X
            rec_y = self.G(fake_x)          # Y -> X -> Y
            # adversarial: fakes should look real to the target discriminator
            adv = self._mse(self.D_Y(fake_y), 1.0) + self._mse(self.D_X(fake_x), 1.0)
            cyc = self.l1(rec_x, x) + self.l1(rec_y, y)
            idt = self.l1(self.G(y), y) + self.l1(self.F(x), x)  # identity on target
            lossG = adv + self.lambda_cyc * cyc + self.lambda_id * idt
            self.optG.zero_grad(); lossG.backward(); self.optG.step()

            # --- Discriminators: real -> 1, fake -> 0 (detached) ---
            lossD = (self._mse(self.D_Y(y), 1.0) + self._mse(self.D_Y(fake_y.detach()), 0.0)
                     + self._mse(self.D_X(x), 1.0) + self._mse(self.D_X(fake_x.detach()), 0.0))
            self.optD.zero_grad(); lossD.backward(); self.optD.step()

            self.g_hist.append(lossG.item()); self.cyc_hist.append(cyc.item())
        return self

    @torch.no_grad()
    def generate(self, X: np.ndarray) -> np.ndarray:
        """Translate domain-X points to domain Y via G."""
        x = torch.as_tensor(X, dtype=torch.float32, device=self.dev)
        return self.G(x).cpu().numpy()

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    X = make_domain_x(1024)
    Y = make_domain_y(1024)
    print(f"domain X mean={X.mean(0).round(2)} std={X.std(0).round(2)}")
    print(f"domain Y mean={Y.mean(0).round(2)} std={Y.std(0).round(2)}")

    gan = CycleGANTorch().fit(X, Y, steps=1500, batch=128)
    c0, c1 = np.mean(gan.cyc_hist[:50]), np.mean(gan.cyc_hist[-50:])
    print(f"cycle-consistency L1: {c0:.3f} -> {c1:.3f} (falling => X->Y->X recovers X)")

    # Quality proxy: translated X should move toward domain Y's distribution.
    fake_y = gan.generate(make_domain_x(800, seed=99))
    print(f"translated G(X) mean={fake_y.mean(0).round(2)} std={fake_y.std(0).round(2)}")
    print(f"target     Y    mean={Y.mean(0).round(2)} std={Y.std(0).round(2)}")
    sep = np.linalg.norm(X.mean(0) - Y.mean(0))
    err = np.linalg.norm(fake_y.mean(0) - Y.mean(0))
    print(f"mean gap |G(X) - Y| = {err:.3f} vs inter-domain gap {sep:.3f} "
          f"(gap shrinks => G translates X toward Y)")

## 6. Train

In [ ]:
demo()

## 7. Visualization

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import cyclegan as M

X = M.make_domain_x(800)
Y = M.make_domain_y(800)
gan = M.CycleGANTorch().fit(X, Y, steps=1500, batch=128)
fake_y = gan.generate(M.make_domain_x(800, seed=99))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(X[:, 0], X[:, 1], s=8, alpha=.4, label="domain X", color="C0")
ax[0].scatter(Y[:, 0], Y[:, 1], s=8, alpha=.4, label="domain Y", color="C1")
ax[0].scatter(fake_y[:, 0], fake_y[:, 1], s=8, alpha=.5, label="G(X) -> Y", color="C3")
ax[0].set_title("Unpaired domains and the learned translation")
ax[0].legend(); ax[0].set_aspect("equal")
ax[1].plot(gan.cyc_hist, alpha=.8)
ax[1].set_xlabel("step"); ax[1].set_ylabel("cycle L1")
ax[1].set_title("Cycle-consistency loss falling (X->Y->X recovers X)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- Cycle-consistency replaces paired supervision: it is the L1 round-trip term
  that makes unpaired translation well-posed and resists mode collapse.
- Identity loss stabilizes color/scale; the LSGAN adversarial loss is steadier
  than BCE for this two-GAN system.
- Pitfalls: too-large $\lambda_{\text{cyc}}$ makes $G$ near-identity (no
  translation); the cycle constraint only enforces invertibility, not
  *correctness*, so geometry-changing tasks (e.g. shape changes) are hard.